# WindOps AI Copilot — EDA: Simulated Fleet Data

This notebook validates the full analytics pipeline using simulated SCADA-like data.

The goal is not exploratory discovery — the data is synthetic. The goal is to verify that:
- The simulation produces physically plausible signals
- Fault injection creates detectable patterns
- The feature, anomaly and risk pipelines behave as expected
- The priority ranking produces operationally coherent results

All data is generated by `src/data_generation.py` using a generic 3 MW turbine model.

In [2]:
# ===============================
# SETUP
# ===============================

import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.append("..")

from src.data_generation import load_demo_scenario
from src.features import build_features, get_feature_columns
from src.anomaly import run_anomaly_pipeline
from src.risk import run_risk_pipeline, summarize_risk_by_turbine
from src.impact import run_impact_pipeline
from src.prioritization import run_prioritization_pipeline
from src.expected_power import expected_power

SCENARIO = "green"
PLOT_STYLE = "seaborn-v0_8-whitegrid"
plt.style.use(PLOT_STYLE)

## 1. Data Generation

We load the **green scenario**: a mostly healthy fleet with a small fraction of turbines affected by simulated faults. This is the baseline operating condition.

In [ ]:
# ===============================
# LOAD AND BUILD PIPELINE
# ===============================

df_hourly = load_demo_scenario(SCENARIO)
df_features = build_features(df_hourly)
df_anomaly, iso_model = run_anomaly_pipeline(df_features)
df_risk, risk_summary = run_risk_pipeline(df_anomaly)
loss_summary = run_impact_pipeline(df_risk)
priority = run_prioritization_pipeline(risk_summary, loss_summary)

print(f"Turbines:   {df_hourly['turbine_id'].nunique()}")
print(f"Timesteps:  {df_hourly['timestamp'].nunique()} hours")
print(f"Date range: {df_hourly['timestamp'].min()} → {df_hourly['timestamp'].max()}")
print(f"Total rows: {len(df_hourly)}")
print(f"\nColumns: {list(df_hourly.columns)}")

## 2. Fleet Overview

Availability per turbine across the simulation period. Values below 1.0 indicate downtime events.

In [ ]:
# ===============================
# AVAILABILITY BY TURBINE
# ===============================

availability = (
    df_hourly.groupby("turbine_id")["availability"]
    .mean()
    .reset_index()
    .sort_values("availability")
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.barh(availability["turbine_id"], availability["availability"], color="steelblue")
ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Mean availability")
ax.set_title("Fleet availability — green scenario")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

print(f"Fleet mean availability: {availability['availability'].mean():.2%}")

## 3. Wind Speed Distribution

Wind speed drives everything downstream. We check that the simulated distribution is physically plausible: a right-skewed distribution centered around the mean wind speed, clipped at cut-out (25 m/s).

In [ ]:
# ===============================
# WIND SPEED DISTRIBUTION
# ===============================

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(df_hourly["wind_speed"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(3.0, color="orange", linestyle="--", label="Cut-in (3 m/s)")
axes[0].axvline(12.0, color="green", linestyle="--", label="Rated (12 m/s)")
axes[0].axvline(25.0, color="red", linestyle="--", label="Cut-out (25 m/s)")
axes[0].set_xlabel("Wind speed (m/s)")
axes[0].set_ylabel("Count")
axes[0].set_title("Wind speed distribution")
axes[0].legend()

# One turbine time series sample
sample = df_hourly[df_hourly["turbine_id"] == "WTG-01"].head(168)
axes[1].plot(sample["timestamp"], sample["wind_speed"], linewidth=0.8, color="steelblue")
axes[1].set_xlabel("Timestamp")
axes[1].set_ylabel("Wind speed (m/s)")
axes[1].set_title("WTG-01 — wind speed (first 7 days)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 4. Power Curve: Expected vs Actual

The theoretical power curve (cubic interpolation) is overlaid on actual power output. Healthy turbines should cluster tightly around the curve. Faulty turbines will show systematic underperformance, especially at mid-to-high wind speeds.

In [ ]:
# ===============================
# POWER CURVE
# ===============================

ws_range = np.linspace(0, 25, 300)
pw_curve = expected_power(ws_range)

fault_turbines = (
    df_hourly[df_hourly["fault_type"] != "none"]["turbine_id"]
    .unique()
)
healthy_turbines = (
    df_hourly[df_hourly["fault_type"] == "none"]["turbine_id"]
    .unique()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, ids, label, color in [
    (axes[0], healthy_turbines, "Healthy turbines", "steelblue"),
    (axes[1], fault_turbines, "Faulty turbines", "tomato"),
]:
    subset = df_hourly[df_hourly["turbine_id"].isin(ids)]
    ax.scatter(
        subset["wind_speed"], subset["power_kw"],
        alpha=0.05, s=2, color=color, label="Actual"
    )
    ax.plot(ws_range, pw_curve, color="black", linewidth=1.5, label="Expected curve")
    ax.set_xlabel("Wind speed (m/s)")
    ax.set_ylabel("Power (kW)")
    ax.set_title(label)
    ax.legend(markerscale=5)

plt.suptitle("Power curve: expected vs actual", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Power Gap by Turbine

Mean hourly power gap (as % of expected output) per turbine. Turbines with faults should consistently show higher gaps. This is the primary signal driving the aerodynamic risk subscore.

In [ ]:
# ===============================
# POWER GAP BY TURBINE
# ===============================

gap_by_turbine = (
    df_features.groupby("turbine_id")["power_gap_pct"]
    .mean()
    .reset_index()
    .sort_values("power_gap_pct", ascending=False)
)

fault_map = (
    df_hourly.groupby("turbine_id")["fault_type"]
    .first()
    .to_dict()
)
gap_by_turbine["fault_type"] = gap_by_turbine["turbine_id"].map(fault_map)
gap_by_turbine["is_faulty"] = gap_by_turbine["fault_type"] != "none"

colors = gap_by_turbine["is_faulty"].map({True: "tomato", False: "steelblue"})

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(gap_by_turbine["turbine_id"], gap_by_turbine["power_gap_pct"], color=colors)
ax.axhline(0.10, color="orange", linestyle="--", linewidth=0.9, label="Low risk threshold (10%)")
ax.axhline(0.25, color="red", linestyle="--", linewidth=0.9, label="High risk threshold (25%)")
ax.set_ylabel("Mean power gap (%)")
ax.set_title("Mean power gap by turbine — green scenario")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.tick_params(axis="x", rotation=45)
ax.legend()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="tomato", label="Faulty"),
    Patch(facecolor="steelblue", label="Healthy"),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][0:2])

plt.tight_layout()
plt.show()

## 6. Sensor Signals: Temperature and Vibration

Gear oil temperature and rotor RPM are the main mechanical signals. Gearbox degradation faults are expected to show elevated temperatures. Pitch malfunction shows up primarily in the power gap at high wind speeds.

In [ ]:
# ===============================
# SENSOR SIGNALS
# ===============================

healthy = df_hourly[df_hourly["fault_type"] == "none"]
faulty = df_hourly[df_hourly["fault_type"] != "none"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gear oil temperature
axes[0].hist(healthy["gear_oil_temp_c"], bins=40, alpha=0.6, color="steelblue", label="Healthy")
axes[0].hist(faulty["gear_oil_temp_c"], bins=40, alpha=0.6, color="tomato", label="Faulty")
axes[0].axvline(65.0, color="orange", linestyle="--", linewidth=0.9, label="Low threshold")
axes[0].axvline(80.0, color="red", linestyle="--", linewidth=0.9, label="High threshold")
axes[0].set_xlabel("Gear oil temperature (°C)")
axes[0].set_ylabel("Count")
axes[0].set_title("Gear oil temperature distribution")
axes[0].legend()

# Rotor RPM
axes[1].hist(healthy["rotor_rpm"], bins=40, alpha=0.6, color="steelblue", label="Healthy")
axes[1].hist(faulty["rotor_rpm"], bins=40, alpha=0.6, color="tomato", label="Faulty")
axes[1].set_xlabel("Rotor RPM")
axes[1].set_ylabel("Count")
axes[1].set_title("Rotor RPM distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Anomaly Detection

Distribution of IsolationForest anomaly scores across the fleet. A score of 1.0 indicates maximum anomalousness. Faulty turbines should show higher scores on average, though IsolationForest is unsupervised and does not use fault labels.

In [ ]:
# ===============================
# ANOMALY SCORES
# ===============================

df_anomaly["fault_type"] = df_anomaly["turbine_id"].map(fault_map)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Score distribution by fault status
for label, subset, color in [
    ("Healthy", df_anomaly[df_anomaly["fault_type"] == "none"], "steelblue"),
    ("Faulty", df_anomaly[df_anomaly["fault_type"] != "none"], "tomato"),
]:
    axes[0].hist(subset["anomaly_score"], bins=40, alpha=0.6, color=color, label=label)

axes[0].set_xlabel("Anomaly score")
axes[0].set_ylabel("Count")
axes[0].set_title("Anomaly score distribution")
axes[0].legend()

# Flagging rate by turbine
flag_rate = (
    df_anomaly.groupby("turbine_id")["anomaly_flag"]
    .mean()
    .reset_index()
    .sort_values("anomaly_flag", ascending=False)
)
flag_rate["is_faulty"] = flag_rate["turbine_id"].map(fault_map) != "none"
bar_colors = flag_rate["is_faulty"].map({True: "tomato", False: "steelblue"})

axes[1].bar(flag_rate["turbine_id"], flag_rate["anomaly_flag"], color=bar_colors)
axes[1].set_ylabel("Anomaly flag rate")
axes[1].set_title("Fraction of hours flagged as anomalous")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Overall flag rate: {df_anomaly['anomaly_flag'].mean():.2%}")

## 8. Risk Scores

The risk score combines three subscores — aerodynamic, mechanical and anomaly — with fixed weights defined in `config.py`. The hybrid design (70% absolute + 30% relative) ensures that a fully degraded fleet does not mask individual critical turbines.

In [ ]:
# ===============================
# RISK SCORE HEATMAP
# ===============================

risk_pivot = (
    df_risk.groupby("turbine_id")[["aero_risk", "mech_risk", "anomaly_risk", "risk_score"]]
    .mean()
    .sort_values("risk_score", ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    risk_pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn_r",
    vmin=0,
    vmax=1,
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Mean risk subscores by turbine (green scenario)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 9. Priority Ranking

Final turbine priority combining risk score, energy loss and criticality. This is the output consumed by the agent to generate action plans.

In [ ]:
# ===============================
# PRIORITY RANKING TABLE
# ===============================

display_cols = [
    "priority_rank",
    "turbine_id",
    "priority_score",
    "risk_score_mean",
    "loss_mwh_total",
    "criticality",
]

print("Top 10 priority turbines:\n")
print(priority[display_cols].head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(
    priority["turbine_id"],
    priority["priority_score"],
    color=priority["priority_score"].apply(
        lambda s: "tomato" if s > 0.6 else "orange" if s > 0.35 else "steelblue"
    ),
)
ax.set_ylabel("Priority score")
ax.set_title("Turbine priority ranking — green scenario")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 10. Simulation Note

The data used in this notebook is synthetically generated by `src/data_generation.py`. It is not a substitute for real SCADA data.

**What this notebook validates:**
- The simulation produces physically plausible wind and power signals
- Fault injection creates detectable patterns in sensor signals and power gap
- The feature engineering pipeline correctly captures degradation signals
- IsolationForest identifies anomalous turbines without using fault labels
- The risk score separates healthy from degraded turbines
- The priority ranking produces operationally coherent results

**Known limitations:**
- Single turbine model (generic 3 MW), no manufacturer-specific curves
- Simplified fault modes: real faults are more complex and correlated
- No meteorological correlation between turbines (wind is simulated independently)
- Anomaly model trained and evaluated on the same dataset — acceptable for demo, not for production

A comparison against the Kelmarsh public SCADA dataset (Zenodo #5841834) is planned for a future notebook to validate signal ranges against real operational data.